In [ ]:
pip install -U langchain langchain_openai newspaper3k gradio


In [ ]:
import os
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage
import newspaper

# 🔑 Set OpenRouter API Key
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# 🤖 Initialize LangChain Chat Model
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)

# 🌟 Chatbot Function
def chatbot_response(user_input):
    system_prompt = "You are a helpful and friendly AI assistant."
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_input)])
    return response.content

# 📰 News Summarizer Function
def summarize_news(url):
    try:
        article = newspaper.Article(url)
        article.download()
        article.parse()

        if not article.text:
            return "❌ Failed to extract article content. Check the URL."

        system_prompt = "You are a news summarization assistant. Summarize the article in simple and concise language with at least 200 words split into 3 paragraphs."
        response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=f"Summarize this article:\n\n{article.text}")])

        return response.content
    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# 🌍 Translator Function
LANGUAGES = {"English": "en", "French": "fr", "Spanish": "es", "German": "de", "Chinese": "zh", "Japanese": "ja", "Korean": "ko", "Hindi": "hi", "Portuguese": "pt", "Russian": "ru", "Arabic": "ar"}

def translate_text(text, source_language, target_language):
    if not text.strip():
        return "❌ Please enter some text to translate."
    if source_language == target_language:
        return "⚠️ Source and target languages are the same. Please select different languages."

    system_prompt = f"You are a professional translator. Translate from {source_language} to {target_language} in a natural and fluent way."
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=text)])

    return response.content

# 📄 Resume Analyzer Function
def analyze_resume(resume_text):
    if not resume_text.strip():
        return "❌ Please enter resume text for analysis."

    system_prompt = (
        "You are an experienced hiring manager. Analyze the resume thoroughly and provide:\n"
        "1️⃣ **Strengths**\n2️⃣ **Weaknesses**\n3️⃣ **Formatting Issues**\n4️⃣ **Skill Gaps**\n5️⃣ **Suggested Rewrites**"
    )

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=f"Analyze this resume:\n\n{resume_text}")])
    return response.content

# 🩺 Medical Diagnosis Function
def diagnose_medical_report(report_text):
    if not report_text.strip():
        return "❌ Please enter the medical report text for analysis."

    system_prompt = (
        "You are an expert medical doctor. Given a patient's medical report, provide:\n"
        "1️⃣ **Possible Diagnosis**\n2️⃣ **Key Observations**\n3️⃣ **Suggested Next Steps**\n\n"
        "⚠️ Disclaimer: AI-generated analysis should not replace professional medical advice."
    )

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=f"Analyze this medical report:\n\n{report_text}")])
    return response.content

# 🎛️ Dynamic Input Handling
def main_tool(mode, input_text, lang_source, lang_target):
    if mode == "Chatbot":
        return chatbot_response(input_text)
    elif mode == "News Summarizer":
        return summarize_news(input_text)
    elif mode == "Translator":
        return translate_text(input_text, lang_source, lang_target)
    elif mode == "Resume Analyzer":
        return analyze_resume(input_text)
    elif mode == "Medical Diagnosis Assistant":
        return diagnose_medical_report(input_text)
    else:
        return "❌ Invalid Mode Selected"

# 🎨 Dynamic UI Function to Show Language Dropdowns Only for Translator
def update_ui(mode):
    if mode == "Translator":
        return gr.update(visible=True), gr.update(visible=True)  # Show language dropdowns
    else:
        return gr.update(visible=False), gr.update(visible=False)  # Hide them

# 🎨 Gradio Interface
with gr.Blocks() as iface:
    gr.Markdown("## 🧠 AI Multi-Tool Suite (OpenRouter + LangChain)")

    # Dropdown to select tool
    mode_select = gr.Dropdown(
        ["Chatbot", "News Summarizer", "Translator", "Resume Analyzer", "Medical Diagnosis Assistant"],
        label="Select AI Tool",
        value="Chatbot",
    )

    # Main input text field
    input_text = gr.Textbox(label="Enter Input (Text or URL)")

    # Language dropdowns (Initially hidden)
    lang_source = gr.Dropdown(choices=list(LANGUAGES.keys()), label="Source Language", value="English", visible=False)
    lang_target = gr.Dropdown(choices=list(LANGUAGES.keys()), label="Target Language", value="French", visible=False)

    # Output text field
    output_text = gr.Textbox(label="AI Output")

    # Button to execute
    submit_button = gr.Button("Run")

    # Link mode selection to dynamic UI updates
    mode_select.change(update_ui, inputs=[mode_select], outputs=[lang_source, lang_target])

    # Link button click to function execution
    submit_button.click(main_tool, inputs=[mode_select, input_text, lang_source, lang_target], outputs=output_text)

# Launch UI
iface.launch(share=True)
